# Web Scraping — Date Drilled
Fetches the **Date Drilled** field from [GeoSteam](https://geosteam.conservation.ca.gov) for each well in the CSV and saves the result to `Data/Processed/geothermal_wells_with_dates.csv`.
This should give us additional data that was not in the CSV file at first

## 1. Imports & Configuration

In [1]:
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL   = "https://geosteam.conservation.ca.gov"
SEARCH_URL = f"{BASE_URL}/GeoWellSearch"

INPUT_CSV  = "../Data/Raw/geothermal_wells_ca.csv"
OUTPUT_CSV = "../Data/Processed/geothermal_wells_with_dates_test.csv"

API_COLUMN = "APINumber"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/135.0.0.0 Safari/537.36 Edg/135.0.0.0"
    )
}

session = requests.Session()
session.headers.update(HEADERS)

print("Setup complete.")

Setup complete.


## 2. Helper Functions

In [2]:
def normalize_api(value):
    """Normalize API number to 8 digits by left-padding with zeros."""
    if pd.isna(value):
        return None
    digits = re.sub(r"\D", "", str(value).strip())
    if not digits:
        return None
    return digits.zfill(8)


def get_with_retry(url, params=None, retries=3):
    """GET request with exponential backoff retry on failure."""
    last_exc: Exception = RuntimeError("No attempts made")
    for attempt in range(retries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            last_exc = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise last_exc


def get_detail_url(api_number):
    """Construct the well detail URL directly from the API number."""
    return f"{BASE_URL}/GeoWellSearch/WellDetails?apinum={api_number}"


def extract_year_drilled(detail_html):
    soup = BeautifulSoup(detail_html, "html.parser")

    # Primary: <li> elements containing "Year Drilled"
    for li in soup.find_all("li"):
        text = li.get_text(" ", strip=True)
        if "Year Drilled" in text:
            b = li.find("b")
            if b:
                value = b.get_text(strip=True)
                return value if value else None
            m = re.search(r"Year Drilled[:\s]+(\d+)", text)
            if m:
                return m.group(1)

    # Secondary: table rows
    for tr in soup.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        if len(cells) >= 2:
            label = cells[0].get_text(" ", strip=True).lower()
            value = cells[1].get_text(" ", strip=True)
            if "year drilled" in label or "date drilled" in label:
                return value if value else None

    # Tertiary: plain text fallback
    full_text = soup.get_text("\n", strip=True)
    m = re.search(r"Year\s*Drilled[:\s]+(\d+)", full_text, re.IGNORECASE)
    if m:
        return m.group(1)

    return None


def fetch_year_drilled(api_number):
    time.sleep(2)  # rate-limit: one request every 2 seconds
    detail_url = get_detail_url(api_number)
    r = get_with_retry(detail_url)

    year_drilled = extract_year_drilled(r.text)
    if not year_drilled:
        return None, "year_not_found"

    return year_drilled, "ok"


def save_output(df, cache, path):
    """Create or update the output CSV with scraped year values."""
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if os.path.exists(path):
        out = pd.read_csv(path, dtype={API_COLUMN: str})
        out["api_norm"] = out[API_COLUMN].apply(normalize_api)
        # Cast to object so string years can be assigned without dtype errors
        out["Year Drilled"] = out["year_drilled"].astype(object) if "year_drilled" in out.columns else pd.Series([None] * len(out), dtype=object)
        out["Year Drilled Status"] = out["year_drilled_status"].astype(object) if "year_drilled_status" in out.columns else pd.Series([None] * len(out), dtype=object)
        for idx, row in out.iterrows():
            key = row["api_norm"]
            if pd.notna(key) and key in cache:
                out.at[idx, "Year Drilled"] = cache[key][0]
                out.at[idx, "Year Drilled Status"] = cache[key][1]
    else:
        out = df.copy()
        out["Year Drilled"] = pd.Series(
            [cache.get(x, (None, None))[0] if pd.notna(x) else None for x in out["api_norm"]],
            dtype=object
        )
        out["Year Drilled Status"] = pd.Series(
            [cache.get(x, (None, None))[1] if pd.notna(x) else None for x in out["api_norm"]],
            dtype=object
        )

    out.to_csv(path, index=False)
    return out


print("Functions defined.")

Functions defined.


## 3. Load Data & Inspect API Numbers

In [3]:
df = pd.read_csv(INPUT_CSV)

if API_COLUMN not in df.columns:
    raise ValueError(f"Column '{API_COLUMN}' not found. Available: {list(df.columns)}")

df["api_norm"] = df[API_COLUMN].apply(normalize_api)

print(f"Total rows: {len(df)}")
print(f"Unique API numbers: {df['api_norm'].nunique()}")
print("\nAPI digit-length distribution after normalization:")
print(df["api_norm"].dropna().str.len().value_counts().sort_index())

df[[API_COLUMN, "api_norm"]].head(10)

Total rows: 4336
Unique API numbers: 4336

API digit-length distribution after normalization:
api_norm
8    4336
Name: count, dtype: int64


,APINumber,api_norm
0,1190001,01190001
1,1190002,01190002
2,1190003,01190003
3,2590001,02590001
4,2590002,02590002
5,2590003,02590003
6,2590004,02590004
7,2590005,02590005
8,2590006,02590006
9,2590007,02590007


## 4. Resume Check
If the output file already exists, previously fetched APIs are skipped.

In [4]:
already_done = {}

if os.path.exists(OUTPUT_CSV):
    done_df = pd.read_csv(OUTPUT_CSV, dtype={API_COLUMN: str})
    # Re-derive api_norm so leading zeros are not lost by pandas inference
    done_df["api_norm"] = done_df[API_COLUMN].apply(normalize_api)
    for _, row in done_df.iterrows():
        key = row["api_norm"]
        if pd.notna(key) and pd.notna(row.get("Year Drilled Status")):
            already_done[key] = (
                row.get("Year Drilled"),
                row.get("Year Drilled Status"),
            )
    print(f"Resuming: {len(already_done)} APIs already processed, will skip them.")
else:
    print("No existing output file found. Starting fresh.")

unique_apis = [a for a in df["api_norm"].dropna().unique() if a not in already_done]
cache = dict(already_done)

print(f"APIs left to fetch: {len(unique_apis)}")

No existing output file found. Starting fresh.
APIs left to fetch: 4336


## 5. Scrape Date Drilled
Runs through the first 10 API numbers for testing

In [5]:
# --- TEST: first 10 API numbers from CSV ---
test_apis = df["api_norm"].dropna().head(10).tolist()

for api in test_apis:
    if api in cache:
        print(f"API {api}: already in cache ({cache[api][1]}), skipping.")
        continue
    print(f"\nAPI: {api}  ->  {get_detail_url(api)}")
    try:
        year_value, status = fetch_year_drilled(api)
        cache[api] = (year_value, status)
        print(f"  Status : {status}")
        print(f"  Year   : {year_value}")
    except Exception as e:
        cache[api] = (None, "error")
        print(f"  Error  : {e}")

# Save results to CSV immediately
save_output(df, cache, OUTPUT_CSV)
print("\nSaved to:", OUTPUT_CSV)


API: 01190001  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=01190001
  Status : ok
  Year   : 1968

API: 01190002  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=01190002
  Status : ok
  Year   : 1965

API: 01190003  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=01190003
  Status : ok
  Year   : 1980

API: 02590001  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=02590001
  Status : ok
  Year   : 1972

API: 02590002  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=02590002
  Status : year_not_found
  Year   : None

API: 02590003  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=02590003
  Status : ok
  Year   : 1972

API: 02590004  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?apinum=02590004
  Status : ok
  Year   : 1971

API: 02590005  ->  https://geosteam.conservation.ca.gov/GeoWellSearch/WellDetails?ap

## 6. Summary

In [6]:
# Save / update the output CSV
result_df = save_output(df, cache, OUTPUT_CSV)
print(f"Saved to: {OUTPUT_CSV}")

# Summary
statuses = result_df["Year Drilled Status"].dropna()
print("\n--- Scrape Summary ---")
print(statuses.value_counts().to_string())

filled = result_df["Year Drilled"].notna().sum()
print(f"\nWells with year: {filled} / {len(result_df)}")

result_df[[API_COLUMN, "Year Drilled", "Year Drilled Status"]].head(10)

Saved to: ../Data/Processed/geothermal_wells_with_dates_test.csv

--- Scrape Summary ---
Year Drilled Status
ok                9
year_not_found    1

Wells with year: 9 / 4336


,APINumber,Year Drilled,Year Drilled Status
0,1190001,1968,ok
1,1190002,1965,ok
2,1190003,1980,ok
3,2590001,1972,ok
4,2590002,None,year_not_found
5,2590003,1972,ok
6,2590004,1971,ok
7,2590005,1973,ok
8,2590006,1972,ok
9,2590007,1972,ok
